In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 6
INTERVAL = "5m"
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "OPUSDT"
INTERVAL = "5m"
TARGET_HORIZON = 6
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split
from models import tune_selected_features_only , make_bucket_table, fit_final_model

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_{INTERVAL}.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")

features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 83,520


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,vol_regime_ratio,hour_sin,hour_cos,dow_sin,dow_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-06-01 00:00:00+00:00,0.650,0.651,0.648,0.648,88549.44,2025-06-01 00:04:59.999999+00:00,57469.22919,220,32406.60,...,NaN,0.0,1.0,-0.781831,0.62349,0.000000,0.000000,0.000000,NaN,NaN
1,2025-06-01 00:05:00+00:00,0.648,0.650,0.648,0.649,24360.96,2025-06-01 00:09:59.999999+00:00,15815.45574,136,7625.72,...,NaN,0.0,1.0,-0.781831,0.62349,0.000080,0.000016,0.000064,NaN,NaN
2,2025-06-01 00:10:00+00:00,0.650,0.650,0.647,0.648,141399.73,2025-06-01 00:14:59.999999+00:00,91577.87596,172,22999.76,...,NaN,0.0,1.0,-0.781831,0.62349,0.000062,0.000025,0.000037,NaN,NaN
3,2025-06-01 00:15:00+00:00,0.647,0.648,0.646,0.647,159778.93,2025-06-01 00:19:59.999999+00:00,103322.46961,398,60700.30,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000033,0.000013,-0.000047,NaN,NaN
4,2025-06-01 00:20:00+00:00,0.648,0.648,0.646,0.648,76141.40,2025-06-01 00:24:59.999999+00:00,49319.07128,143,47804.41,...,NaN,0.0,1.0,-0.781831,0.62349,-0.000027,0.000005,-0.000033,NaN,NaN


In [8]:
target_col = f"target_{TARGET_HORIZON}"
ret_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col, ret_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col, ret_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]
fwd_ret_train = train_df[ret_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]
fwd_ret_valid = valid_df[ret_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]
fwd_ret_test = test_df[ret_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 83,330
[info] optuna train rows: 53,331
[info] valid rows:        13,333
[info] test rows:         16,666


In [9]:
results = tune_selected_features_only(
    model_type=MODEL_TYPE,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    fwd_ret_valid=fwd_ret_valid,
    n_trials=100,
    objective_metric="roc_auc",
    top_k=25,
)

print(results["selected_features"])
print(results["feature_importance"].head(30))

[I 2026-03-23 15:12:08,429] A new study created in memory with name: no-name-dad8e02f-b4d1-4ae4-9d31-e791bbd92e09


[I 2026-03-23 15:12:08,596] Trial 0 finished with value: 0.528658432763526 and parameters: {'n_estimators': 500, 'learning_rate': 0.046187109390049115, 'max_depth': 5, 'subsample': 0.7996646210492592, 'colsample_bytree': 0.6890046601106091, 'colsample_bylevel': 0.6889986300840507, 'min_child_weight': 5, 'gamma': 2.5985284373248057, 'reg_alpha': 0.12306931514988033, 'reg_lambda': 8.341106432362084, 'scale_pos_weight': 1.0468103659428847}. Best is trial 0 with value: 0.528658432763526.


[I 2026-03-23 15:12:08,763] Trial 1 finished with value: 0.5281736512038097 and parameters: {'n_estimators': 900, 'learning_rate': 0.03818145165896871, 'max_depth': 3, 'subsample': 0.6954562418017751, 'colsample_bytree': 0.6958511274633585, 'colsample_bylevel': 0.7260605607398845, 'min_child_weight': 13, 'gamma': 1.2958350559263474, 'reg_alpha': 0.010295300642650052, 'reg_lambda': 6.252287916406214, 'scale_pos_weight': 1.074705719319472}. Best is trial 0 with value: 0.528658432763526.


[I 2026-03-23 15:12:09,025] Trial 2 finished with value: 0.5272244598949057 and parameters: {'n_estimators': 500, 'learning_rate': 0.018033330377234345, 'max_depth': 4, 'subsample': 0.8462939903482534, 'colsample_bytree': 0.6999184455395899, 'colsample_bylevel': 0.778558609603403, 'min_child_weight': 14, 'gamma': 0.13935123815999317, 'reg_alpha': 0.12957079329680446, 'reg_lambda': 1.6666983286066417, 'scale_pos_weight': 1.0571562747820513}. Best is trial 0 with value: 0.528658432763526.


[I 2026-03-23 15:12:09,175] Trial 3 finished with value: 0.5290670507642108 and parameters: {'n_estimators': 900, 'learning_rate': 0.047309442068985116, 'max_depth': 5, 'subsample': 0.7261534422933427, 'colsample_bytree': 0.6744180285015959, 'colsample_bylevel': 0.8210582566280392, 'min_child_weight': 12, 'gamma': 0.3661147045343365, 'reg_alpha': 0.05269751777340593, 'reg_lambda': 1.1085122517311703, 'scale_pos_weight': 1.274187521114886}. Best is trial 3 with value: 0.5290670507642108.


...

[I 2026-03-23 15:12:28,878] Trial 96 finished with value: 0.5320000962539909 and parameters: {'n_estimators': 700, 'learning_rate': 0.04267919928870296, 'max_depth': 5, 'subsample': 0.8786364958334091, 'colsample_bytree': 0.7504313100260636, 'colsample_bylevel': 0.8933500973858509, 'min_child_weight': 20, 'gamma': 0.8963630856981646, 'reg_alpha': 1.0160866372817225, 'reg_lambda': 1.2500764514431937, 'scale_pos_weight': 1.0734885715751386}. Best is trial 72 with value: 0.5354866134469437.


[I 2026-03-23 15:12:29,064] Trial 97 finished with value: 0.5348211643167278 and parameters: {'n_estimators': 800, 'learning_rate': 0.04969404918258262, 'max_depth': 5, 'subsample': 0.8642576896037106, 'colsample_bytree': 0.737157009195606, 'colsample_bylevel': 0.841036008876654, 'min_child_weight': 19, 'gamma': 0.5636282497135507, 'reg_alpha': 0.6792851070792282, 'reg_lambda': 1.4115734298658067, 'scale_pos_weight': 1.1352235863613438}. Best is trial 72 with value: 0.5354866134469437.


[I 2026-03-23 15:12:29,236] Trial 98 finished with value: 0.5262953139973563 and parameters: {'n_estimators': 800, 'learning_rate': 0.04931315139344371, 'max_depth': 5, 'subsample': 0.888358863989115, 'colsample_bytree': 0.7077728273932529, 'colsample_bylevel': 0.8445041832056518, 'min_child_weight': 18, 'gamma': 0.5349598727829761, 'reg_alpha': 1.308239874387396, 'reg_lambda': 1.4153562247792935, 'scale_pos_weight': 1.1345997240913095}. Best is trial 72 with value: 0.5354866134469437.


[I 2026-03-23 15:12:29,433] Trial 99 finished with value: 0.5279122279755517 and parameters: {'n_estimators': 900, 'learning_rate': 0.04984654775134686, 'max_depth': 5, 'subsample': 0.8950261683857356, 'colsample_bytree': 0.6501435071088626, 'colsample_bylevel': 0.877535834415719, 'min_child_weight': 20, 'gamma': 1.5063188304709598, 'reg_alpha': 1.8055871351753052, 'reg_lambda': 1.956145126544781, 'scale_pos_weight': 1.1073039212952616}. Best is trial 72 with value: 0.5354866134469437.


['dist_ma_30', 'vol_30', 'dow_cos', 'dow_sin', 'hour_sin', 'atr_norm', 'mom_60', 'close_pos_in_bar', 'hour_cos', 'macd_hist', 'vol_regime_ratio', 'imbalance_15', 'trend_strength', 'dist_ma_15', 'range_ratio', 'mom_15', 'bar_range', 'vol_ratio_5_30', 'vol_5', 'mom_5', 'co_spread', 'taker_buy_ratio', 'trades_z', 'volume_z', 'num_trades_mom_5']
feature
dist_ma_30          10.092620
vol_30              10.016239
dow_cos              9.995918
dow_sin              9.827868
hour_sin             9.527457
atr_norm             9.303903
mom_60               9.127124
close_pos_in_bar     9.064037
hour_cos             9.059320
macd_hist            8.718608
vol_regime_ratio     8.617092
imbalance_15         8.566305
trend_strength       8.500024
dist_ma_15           8.295653
range_ratio          8.115604
mom_15               8.097733
bar_range            7.970264
vol_ratio_5_30       7.912795
vol_5                7.737396
mom_5                7.729101
co_spread            7.282583
taker_buy_ratio   

In [10]:
artifacts = fit_final_model(
    model_type=MODEL_TYPE,
    best_params=results["best_params"],
    selected_features=results["selected_features"],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

In [11]:
base_model = artifacts["base_model"]
selected_features = artifacts["selected_features"]

X_train_sel = X_train[selected_features].copy()
X_valid_sel = X_valid[selected_features].copy()
X_train_full_sel = pd.concat([X_train_sel, X_valid_sel], axis=0)

X_test_sel = X_test[selected_features].copy()
y_train_full = pd.concat([y_train, y_valid], axis=0)

train_pred = base_model.predict_proba(X_train_full_sel)[:, 1]
test_pred = base_model.predict_proba(X_test_sel)[:, 1]

In [12]:
train_pred_label = (train_pred >= 0.5).astype(int)
test_pred_label = (test_pred >= 0.5).astype(int)

print("[eval] computing metrics...")

train_ic = spearmanr(train_pred, fwd_ret_train)[0]
test_ic = spearmanr(test_pred, fwd_ret_test)[0]

train_auc = roc_auc_score(y_train_full, train_pred)
test_auc = roc_auc_score(y_test, test_pred)

train_pr_auc = average_precision_score(y_train_full, train_pred)
test_pr_auc = average_precision_score(y_test, test_pred)

train_logloss = log_loss(y_train_full, np.clip(train_pred, 1e-8, 1 - 1e-8))
test_logloss = log_loss(y_test, np.clip(test_pred, 1e-8, 1 - 1e-8))

train_brier = brier_score_loss(y_train_full, train_pred)
test_brier = brier_score_loss(y_test, test_pred)

train_acc = accuracy_score(y_train_full, train_pred_label)
test_acc = accuracy_score(y_test, test_pred_label)

train_precision = precision_score(y_train_full, train_pred_label, zero_division=0)
test_precision = precision_score(y_test, test_pred_label, zero_division=0)

train_recall = recall_score(y_train_full, train_pred_label, zero_division=0)
test_recall = recall_score(y_test, test_pred_label, zero_division=0)

train_f1 = f1_score(y_train_full, train_pred_label, zero_division=0)
test_f1 = f1_score(y_test, test_pred_label, zero_division=0)

print("\n===== RESULTS =====")
print(f"Train IC:        {train_ic:.6f}")
print(f"Test IC:         {test_ic:.6f}")
print(f"Train ROC AUC:   {train_auc:.6f}")
print(f"Test ROC AUC:    {test_auc:.6f}")
print(f"Train PR AUC:    {train_pr_auc:.6f}")
print(f"Test PR AUC:     {test_pr_auc:.6f}")
print(f"Train Log Loss:  {train_logloss:.6f}")
print(f"Test Log Loss:   {test_logloss:.6f}")
print(f"Train Brier:     {train_brier:.6f}")
print(f"Test Brier:      {test_brier:.6f}")
print(f"Train Accuracy:  {train_acc:.6f}")
print(f"Test Accuracy:   {test_acc:.6f}")
print(f"Train Precision: {train_precision:.6f}")
print(f"Test Precision:  {test_precision:.6f}")
print(f"Train Recall:    {train_recall:.6f}")
print(f"Test Recall:     {test_recall:.6f}")
print(f"Train F1:        {train_f1:.6f}")
print(f"Test F1:         {test_f1:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:        0.150089
Test IC:         0.046628
Train ROC AUC:   0.592271
Test ROC AUC:    0.530376
Train PR AUC:    0.559260
Test PR AUC:     0.483355
Train Log Loss:  0.684979
Test Log Loss:   0.690078
Train Brier:     0.245928
Test Brier:      0.248468
Train Accuracy:  0.569543
Test Accuracy:   0.531381
Train Precision: 0.568801
Test Precision:  0.484964
Train Recall:    0.337472
Test Recall:     0.315796
Train F1:        0.423613
Test F1:         0.382511


In [13]:
eval_df = pd.DataFrame({
    "pred": test_pred,
    "y_cls": y_test.values,
    "fwd_ret_test": fwd_ret_test.values,   # continuous realised return
})

eval_df["pred_bin"] = pd.qcut(eval_df["pred"], 10, duplicates="drop")
bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])
print(bucket_stats)

                    mean  count       std
pred_bin                                 
(0.387, 0.457] -0.000660   1667  0.008042
(0.457, 0.467] -0.000499   1667  0.007440
(0.467, 0.475] -0.000516   1666  0.007921
(0.475, 0.481] -0.000269   1667  0.007499
(0.481, 0.487] -0.000286   1666  0.007338
(0.487, 0.493] -0.000295   1667  0.007864
(0.493, 0.5]    0.000100   1666  0.007579
(0.5, 0.507]   -0.000390   1667  0.008317
(0.507, 0.516] -0.000212   1666  0.008589
(0.516, 0.627]  0.000010   1667  0.010468


/tmp/ipykernel_1491458/3344132490.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_stats = eval_df.groupby("pred_bin")["fwd_ret_test"].agg(["mean", "count", "std"])


In [14]:
top_decile_threshold = float(np.quantile(test_pred, 0.9))
bottom_decile_threshold = float(np.quantile(test_pred, 0.1))

top_decile_mean_ret = float(eval_df.loc[eval_df["pred"] >= top_decile_threshold, "fwd_ret_test"].mean())
bottom_decile_mean_ret = float(eval_df.loc[eval_df["pred"] <= bottom_decile_threshold, "fwd_ret_test"].mean())
overall_mean_ret = float(eval_df["fwd_ret_test"].mean())

signal_threshold = 0.6
signal_rate = float((eval_df["pred"] >= signal_threshold).mean())
signal_mean_ret = float(eval_df.loc[eval_df["pred"] >= signal_threshold, "fwd_ret_test"].mean())

In [15]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/OPUSDT__6_predictions.csv


In [16]:
# save model
joblib.dump(artifacts, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(selected_features, f, indent=2)

# save feature importance
results["feature_importance"].to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(results["study"].best_value),
    "model_params": results["best_params"],
    "n_features": int(len(selected_features)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": float(train_ic),
    "test_ic": float(test_ic),
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_pr_auc": float(train_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "train_logloss": float(train_logloss),
    "test_logloss": float(test_logloss),
    "train_brier": float(train_brier),
    "test_brier": float(test_brier),
    "train_accuracy": float(train_acc),
    "test_accuracy": float(test_acc),
    "train_precision": float(train_precision),
    "test_precision": float(test_precision),
    "train_recall": float(train_recall),
    "test_recall": float(test_recall),
    "train_f1": float(train_f1),
    "test_f1": float(test_f1),
    "test_top_decile_threshold": top_decile_threshold,
    "test_bottom_decile_threshold": bottom_decile_threshold,
    "test_top_decile_mean_fwd_ret": top_decile_mean_ret,
    "test_bottom_decile_mean_fwd_ret": bottom_decile_mean_ret,
    "test_overall_mean_fwd_ret": overall_mean_ret,
    "test_signal_threshold": signal_threshold,
    "test_signal_rate": signal_rate,
    "test_signal_mean_fwd_ret": signal_mean_ret,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat(),
    "train_positive_rate": float(y_train_full.mean()),
    "test_positive_rate": float(y_test.mean()),
    "test_pred_mean": float(np.mean(test_pred)),
    "test_pred_std": float(np.std(test_pred)),
    "test_pred_p10": float(np.quantile(test_pred, 0.10)),
    "test_pred_p50": float(np.quantile(test_pred, 0.50)),
    "test_pred_p90": float(np.quantile(test_pred, 0.90)),
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/OPUSDT__h6_model.joblib
[saved] features -> models/xgb/OPUSDT__h6_feature_cols.json
[saved] feature importance -> models/xgb/OPUSDT__h6_feature_importance.csv
[saved] metadata -> models/xgb/OPUSDT__h6_meta.json
